# Chaos Test — AVD Session Host

**Ziel:** Den AVD Session Host gezielt negativ beeinflussen (VM-Shutdown-Fault) via Azure Chaos Studio.

**Aufteilung:**
- Chaos Studio (Target + Capability + Experiment + Rollenzuweisung) wird via Bicep über die GitHub Action **Deploy Chaos Studio** (`infra/chaos/main.bicep`) deployed.
- Das Experiment wird **hier im Notebook** gestartet, überwacht und gestoppt.

**Fault:** `urn:csci:microsoft:virtualMachine:shutdown/1.0` — fährt `vm-avd-cptdazavdvwan` für die Experiment-Dauer (`PT10M`) herunter.

## Variablen

In [ ]:
export PREFIX=cptdazavdvwan
export RG=rg-${PREFIX}
export SUB=$(az account show --query id -o tsv)
export VM=vm-avd-${PREFIX}
export HP=hp-${PREFIX}
export EXPERIMENT=exp-shutdown-${PREFIX}
export APIV=2024-01-01
echo "RG=$RG SUB=$SUB VM=$VM HP=$HP EXPERIMENT=$EXPERIMENT" 

## 0. Voraussetzung: Chaos Studio deployed

Das Experiment muss zuvor über die GitHub Action **Deploy Chaos Studio** (`infra/chaos/main.bicep`) angelegt worden sein. Diese Zelle prüft, ob das Experiment existiert.

In [ ]:
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}?api-version=${APIV}" \
  --query "{name:name, provisioningState:properties.provisioningState, principalId:identity.principalId}" -o json

## 1. Baseline — Session Host Status (vorher)

Erwartung: VM `VM running`, Session Host `Available`.

In [ ]:
echo "=== VM Power State ==="
az vm get-instance-view -g $RG -n $VM \
  --query "instanceView.statuses[?starts_with(code,'PowerState')].displayStatus" -o tsv
echo ""
echo "=== AVD Session Host Status ==="
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/hostPools/${HP}/sessionHosts?api-version=2024-04-03" \
  --query "value[].{name:name,status:properties.status,lastHeartBeat:properties.lastHeartBeat}" -o table

## 2. Experiment starten

Startet das Chaos-Experiment. Danach lesen wir die jüngste Execution aus.

In [ ]:
echo "=== Starte Experiment $EXPERIMENT ==="
az rest --method post \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/start?api-version=${APIV}" \
  -o json
echo ""
echo "Warte, bis die Execution erscheint..."
sleep 10
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/executions?api-version=${APIV}" \
  --query "reverse(sort_by(value,&properties.startedAt))[0].{id:name,status:properties.status,startedAt:properties.startedAt}" -o json

## 3. Experiment-Status überwachen

Mehrfach ausführbar. Status-Werte u.a.: `Running`, `Success`, `Failed`, `Cancelled`.

In [ ]:
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/executions?api-version=${APIV}" \
  --query "reverse(sort_by(value,&properties.startedAt))[0:3].{id:name,status:properties.status,startedAt:properties.startedAt,stoppedAt:properties.stoppedAt}" -o table

## 4. Auswirkung beobachten — Session Host während Shutdown

Erwartung nach ~1–2 Min: VM `VM stopped`/`deallocating`, Session Host `Unavailable`.

In [ ]:
echo "=== VM Power State (während Experiment) ==="
az vm get-instance-view -g $RG -n $VM \
  --query "instanceView.statuses[?starts_with(code,'PowerState')].displayStatus" -o tsv
echo ""
echo "=== AVD Session Host Status (während Experiment) ==="
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/hostPools/${HP}/sessionHosts?api-version=2024-04-03" \
  --query "value[].{name:name,status:properties.status,lastHeartBeat:properties.lastHeartBeat}" -o table

## 5. Experiment vorzeitig stoppen (optional)

Bricht das laufende Experiment ab. Chaos Studio leitet danach die Wiederherstellung ein (VM wird wieder gestartet).

In [ ]:
az rest --method post \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/cancel?api-version=${APIV}" \
  -o json
echo "Cancel ausgelöst." 

## 6. Recovery prüfen

Nach Ablauf der Dauer (oder nach Cancel) wird die VM wieder gestartet. Falls die VM noch aus ist, hier manuell starten.

In [ ]:
STATE=$(az vm get-instance-view -g $RG -n $VM --query "instanceView.statuses[?starts_with(code,'PowerState')].code" -o tsv)
echo "Power State: $STATE"
if [ "$STATE" != "PowerState/running" ]; then
  echo "VM ist nicht running — starte VM..."
  az vm start -g $RG -n $VM
fi
echo ""
echo "=== Finaler Status ==="
az vm get-instance-view -g $RG -n $VM \
  --query "instanceView.statuses[?starts_with(code,'PowerState')].displayStatus" -o tsv
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/hostPools/${HP}/sessionHosts?api-version=2024-04-03" \
  --query "value[].{name:name,status:properties.status}" -o table